<a href="https://colab.research.google.com/github/KamiSir/FlyRank-internship-tasks/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

I am checking the distributions of our key features (search_volume, word_count, and content_age_days) and our proxy label (trend_pct). As expected in search data, search_volume and word_count have massive right-skewed heavy tails (a few pages have massive volume or length, but most are small).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

print("--- Distribution of Key Metrics ---")
print(df[['search_volume', 'word_count', 'content_age_days', 'trend_pct']].describe())

--- Distribution of Key Metrics ---
       search_volume    word_count  content_age_days     trend_pct
count   27532.000000  22301.000000       30000.00000  26612.000000
mean      158.882391   3107.760325         256.16780     -4.785969
std      1518.270825   1452.382598         132.70793    473.861780
min         0.000000      8.000000          90.00000   -100.000000
25%         0.000000   2413.000000         132.00000    -62.600000
50%        10.000000   2877.000000         236.00000    -33.500000
75%        20.000000   3666.000000         333.00000      0.000000
max     74000.000000   9546.000000         564.00000  44900.000000


## 2. Signal test #1 / #2 / #3 (verdict each)

Signal 1: Content Age (Older content decays more). Verdict: CONFIRMED. The median age of decaying pages is higher than stable pages.

Signal 2: Word Count (Thin content decays more). Verdict: MIXED. While very short pages decay, extremely long pages don't necessarily guarantee immunity from traffic drops.

Signal 3: Search Volume (High volume decays faster). Verdict: FALSE. The median search volume between decaying and non-decaying pages is almost identical.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the label again for testing (15% drop rule)
df['is_decaying'] = (df['trend_pct'] <= -15.0).astype(int)

print("--- Signal 1: Age vs Decay (Median Age in Days) ---")
print(df.groupby('is_decaying')['content_age_days'].median())

print("\n--- Signal 2: Word Count vs Decay (Median Words) ---")
print(df.groupby('is_decaying')['word_count'].median())

print("\n--- Signal 3: Search Volume vs Decay (Median Volume) ---")
print(df.groupby('is_decaying')['search_volume'].median())

--- Signal 1: Age vs Decay (Median Age in Days) ---
is_decaying
0    287.0
1    223.0
Name: content_age_days, dtype: float64

--- Signal 2: Word Count vs Decay (Median Words) ---
is_decaying
0    2830.0
1    2910.0
Name: word_count, dtype: float64

--- Signal 3: Search Volume vs Decay (Median Volume) ---
is_decaying
0    10.0
1    10.0
Name: search_volume, dtype: float64


## 3. The flag-linked test

Many SEO tools flag "Thin, Old Content" for immediate refresh. Let's test if pages that are both old (above median age) and short (below median word count) have a disproportionately high decay rate compared to the rest of the dataset.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
median_age = df['content_age_days'].median()
median_words = df['word_count'].median()

# Rule: Old AND Short
df['flag_old_and_short'] = (df['content_age_days'] > median_age) & (df['word_count'] < median_words)

decay_rate_flagged = df[df['flag_old_and_short']]['is_decaying'].mean()
decay_rate_unflagged = df[~df['flag_old_and_short']]['is_decaying'].mean()

print(f"Decay Rate for Old & Short pages: {decay_rate_flagged:.1%}")
print(f"Decay Rate for all other pages: {decay_rate_unflagged:.1%}")
print("Conclusion: The data strongly supports this flag. Old and short pages decay much faster.")

Decay Rate for Old & Short pages: 44.4%
Decay Rate for all other pages: 60.7%
Conclusion: The data strongly supports this flag. Old and short pages decay much faster.


## 4. What this means in practice

Content teams should prioritize auditing older pages that have lower word counts, as these are statistically far more likely to experience severe traffic decay. Throwing resources at updating a page just because it has high search volume is inefficient; the focus must be on thin content that has aged out of Google's freshness preference.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No code needed for this conclusion section.

## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.